#### P4-03
# Random Forest Modelling
#### M03

## 1. Objective

This notebook trains a Random Forest on the training split only.

The flow is `hotel_bookings_train.csv`, then `add_features()`, then shared preprocessing, then a model-specific pipeline. Five-fold stratified cross-validation stays inside the training set. The validation and test splits are not loaded. Tuning uses `scoring="roc_auc"` only.

In [11]:
from pathlib import Path
import sys

import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.features import add_features
from src.preprocessing import SharedPreprocessor, split_classifier_data

pd.set_option("display.max_columns", None)

SPLIT_DIR = ROOT / "data" / "processed" / "split"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

## 2. Training data

Feature engineering and shared preprocessing are fit on the training split. Shared preprocessing drops leakage columns, removes invalid bookings, and groups rare categories using training frequencies only.

In [12]:
train_raw = pd.read_csv(SPLIT_DIR / "hotel_bookings_train.csv")
train_fe = add_features(train_raw)

shared_prep = SharedPreprocessor(rare_threshold=0.01, rare_min_unique=15)
train_prepared = shared_prep.fit_transform(train_fe)

X_train, y_train, booking_ids = split_classifier_data(train_prepared)

assert "booking_id" not in X_train.columns
assert "is_canceled" not in X_train.columns
assert "country" not in X_train.columns

print("Raw train:", train_raw.shape)
print("After features:", train_fe.shape)
print("After shared prep:", train_prepared.shape)
print("X_train:", X_train.shape)
print("Cancellation rate:", round(y_train.mean(), 4))

Raw train: (58990, 37)
After features: (58990, 42)
After shared prep: (58192, 29)
X_train: (58192, 27)
Cancellation rate: 0.2533


In [13]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [col for col in X_train.columns if col not in numeric_features]

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

Numeric: ['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'adr', 'total_nights', 'total_guests', 'has_kids']
Categorical: ['hotel', 'arrival_date_month', 'meal', 'market_segment', 'distribution_channel', 'reserved_room_type', 'deposit_type', 'agent', 'company', 'season', 'arrival_day_of_week']


## 3. Cross-validation

Folds are drawn from the training set only. The same folds are used for the dummy benchmark, the untuned forest, and the search.

In [14]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 4. Dummy benchmark

A prior dummy classifier scores about 0.5 ROC-AUC. It shows whether the forest ranks cancellations better than the class balance alone.

In [15]:
dummy_results = cross_validate(
    DummyClassifier(strategy="prior", random_state=42),
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1,
)
dummy_roc_auc = dummy_results["test_score"].mean()
print("Dummy mean CV ROC-AUC:", round(dummy_roc_auc, 4))

Dummy mean CV ROC-AUC: 0.5


## 5. Random Forest pipeline

Categorical columns are one-hot encoded inside the pipeline. Numeric columns are left unscaled because a tree splits on a threshold, so the unit of a feature does not change the split.

In [16]:
preprocessor = ColumnTransformer([
    ("numeric", "passthrough", numeric_features),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_features,
    ),
])

forest_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        RandomForestClassifier(
            n_estimators=200,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

In [17]:
baseline_results = cross_validate(
    forest_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1,
)
baseline_roc_auc = baseline_results["test_score"].mean()
print("Untuned mean CV ROC-AUC:", round(baseline_roc_auc, 4))
print("Untuned ROC-AUC std:", round(baseline_results["test_score"].std(), 4))

Untuned mean CV ROC-AUC: 0.7897
Untuned ROC-AUC std: 0.0053


## 6. Tuning

`n_estimators` stays at 200. Extra trees mostly stabilise the average, so the search is spent on how deep each tree grows and whether cancellation cases are up-weighted.

The search score is ROC-AUC. Accuracy, precision, and recall are not used to choose a setting.

In [18]:
param_grid = {
    "model__max_depth": [10, 16, None],
    "model__min_samples_leaf": [5, 20],
    "model__class_weight": [None, "balanced"],
}

grid_search = GridSearchCV(
    estimator=forest_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=1,
    refit=True,
    return_train_score=True,
)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best mean CV ROC-AUC:", round(grid_search.best_score_, 4))

Best parameters: {'model__class_weight': 'balanced', 'model__max_depth': None, 'model__min_samples_leaf': 5}
Best mean CV ROC-AUC: 0.8025


In [19]:
tuning_results = pd.DataFrame(grid_search.cv_results_)
tuning_table = tuning_results[
    [
        "param_model__max_depth",
        "param_model__min_samples_leaf",
        "param_model__class_weight",
        "mean_test_score",
        "std_test_score",
        "mean_train_score",
        "rank_test_score",
    ]
].copy()
tuning_table = tuning_table.rename(columns={
    "param_model__max_depth": "Max depth",
    "param_model__min_samples_leaf": "Min samples leaf",
    "param_model__class_weight": "Class weight",
    "mean_test_score": "Mean CV ROC-AUC",
    "std_test_score": "CV ROC-AUC std",
    "mean_train_score": "Mean train ROC-AUC",
    "rank_test_score": "Rank",
})
tuning_table = tuning_table.sort_values("Rank").reset_index(drop=True)
tuning_table

,Max depth,Min samples leaf,Class weight,Mean CV ROC-AUC,CV ROC-AUC std,Mean train ROC-AUC,Rank
0,None,5,balanced,0.802511,0.004333,0.919972,1
1,None,5,NaN,0.802157,0.004355,0.918731,2
2,16,5,balanced,0.797885,0.003500,0.864562,3
3,16,5,NaN,0.797236,0.003586,0.864109,4
4,None,20,NaN,0.794027,0.003554,0.843912,5
5,None,20,balanced,0.793328,0.003448,0.842481,6
6,16,20,NaN,0.791668,0.003299,0.829470,7
7,16,20,balanced,0.790643,0.003274,0.828294,8
8,10,5,balanced,0.783608,0.003404,0.807037,9
9,10,5,NaN,0.783359,0.003712,0.806036,10


In [20]:
comparison = pd.DataFrame({
    "Model": [
        "DummyClassifier",
        "Untuned Random Forest",
        "Tuned Random Forest",
    ],
    "Mean CV ROC-AUC": [
        dummy_roc_auc,
        baseline_roc_auc,
        grid_search.best_score_,
    ],
})
comparison

,Model,Mean CV ROC-AUC
0,DummyClassifier,0.500000
1,Untuned Random Forest,0.789745
2,Tuned Random Forest,0.802511


## 7. Tuning result

The search selected unlimited depth, at least 5 bookings in each leaf, and `class_weight="balanced"`. Mean training-set ROC-AUC was 0.8025, compared with 0.7897 for the untuned forest and 0.5000 for the dummy classifier. The tuned forest ranks cancellations better than either reference. The gain over the untuned forest is small, so most of the ranking comes from using a forest rather than from the particular setting.

## 8. Why Random Forest suits the problem

Cancellation risk depends on combinations of lead time, booking channel, and stay details, and a forest can capture those interactions without assuming a straight-line effect. Averaging many trees gives a cancellation probability for each booking, which is what a deposit decision needs, and that average is more stable than one deep tree. The search keeps the forest that ranks likely cancellations highest on training-set ROC-AUC.

## 9. Save the fitted pipeline

In [ ]:
final_forest = grid_search.best_estimator_
model_path = MODEL_DIR / "random_forest_classifier.joblib"
joblib.dump(final_forest, model_path)
print("Saved:", model_path)